In [0]:
from pyspark.sql.functions import col, explode, arrays_zip, current_timestamp

In [0]:
weather_df=spark.table('pricing_analytics.bronze.weather_data')

In [0]:
df=weather_df.select('latitude', 'longitude', col('daily_units.rain_sum').alias('unitOfRainFall'), col('daily_units.temperature_2m_max').alias('unitOfTemparature'), 'daily', 'MARKET_NAME')

In [0]:
df1=df.withColumn("exploded_data", explode(arrays_zip('daily.rain_sum', 'daily.temperature_2m_max', 'daily.temperature_2m_min', 'daily.time')))

In [0]:
Final_df=df1.select('MARKET_NAME', col('exploded_data.time').alias('weatherDate'), 'unitOfTemparature', col('exploded_data.temperature_2m_max').alias('maximumTemparature'), col('exploded_data.temperature_2m_min').alias('minimumTemparature'), col('unitOfRainFall'), col('exploded_data.rain_sum').alias('rainFall'), 'latitude', 'longitude').withColumn('lake_house_ingestion_date', current_timestamp()).withColumn('lake_house_updated_date', current_timestamp())

In [0]:
Final_df.write.format('delta').mode('overwrite').saveAsTable('pricing_analytics.silver.weather_date')